# ДЗ №4. Задание 3
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Факторизация матрицы, ALS.


## Подготовка

In [1]:
from itertools import islice
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[4]").appName("HW4-ALS")
         .config("spark.driver.memory", "3g").getOrCreate())
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating")

26/11/24 16:49:14 WARN Utils: Your hostname, MacBook-Air-mac-4.local resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/11/24 16:49:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 16:49:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Для сравнения с заданием 2 используем те же строки train и test. Читаем CSV по порядку и повторяем разбиение 80/20 с тем же генератором. Временные файлы нужны, чтобы Spark прочитал оба набора как DataFrame.

In [2]:
def split_ratings(folder, temp_dir):
    rng = np.random.default_rng(42)
    train_path = Path(temp_dir) / "train.csv"
    test_path = Path(temp_dir) / "test.csv"
    with open(Path(folder) / "ratings.csv") as source, train_path.open("w") as train, test_path.open("w") as test:
        next(source)
        while batch := list(islice(source, 100_000)):
            train_mask = rng.random(len(batch), dtype=np.float32) < 0.8
            for row, in_train in zip(batch, train_mask):
                (train if in_train else test).write(row)

    schema = "userId INT, movieId INT, rating FLOAT, timestamp LONG"
    return (spark.read.csv(str(train_path), schema=schema).drop("timestamp"),
            spark.read.csv(str(test_path), schema=schema).drop("timestamp"))

## Малый набор MovieLens

### 1. Выберите модель ALS по минимальному значению rmse. Для этого используйте кросс-валидацию k-folds с k=4

Перебираем 15 сочетаний числа факторов и регуляризации. На каждом фолде исключаем из оценки строки, для которых ALS не знает пользователя или фильм. Все модели проверяются на одних и тех же фолдах.

In [4]:
def train_als(train):
    als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
              seed=42, maxIter=5, numUserBlocks=4, numItemBlocks=4,
              coldStartStrategy="drop")
    grid = (ParamGridBuilder()
            .addGrid(als.rank, [5, 10, 15])
            .addGrid(als.regParam, [0.001, 0.01, 0.1, 1.0, 10.0])
            .build())
    cv = CrossValidator(estimator=als, estimatorParamMaps=grid, evaluator=evaluator,
                        numFolds=4, seed=42)
    result = cv.fit(train)
    best = grid[np.argmin(result.avgMetrics)]
    print(f"Число факторов: {best[als.rank]}, регуляризация: {best[als.regParam]}")
    print(f"Средний RMSE по фолдам: {min(result.avgMetrics):.4f}")
    return result.bestModel


def run_als(folder):
    with TemporaryDirectory() as temp_dir:
        train, test = split_ratings(folder, temp_dir)
        train.cache()
        try:
            mean_rating = train.agg(F.avg("rating")).first()[0]
            model = train_als(train)
            # На итоговом тесте учитываем все оценки, включая новые фильмы и пользователей.
            predictions = model.setColdStartStrategy("nan").transform(test)
            predictions = predictions.fillna({"prediction": mean_rating})
            return evaluator.evaluate(predictions)
        finally:
            train.unpersist()


small_rmse = run_als("../ml-latest-small")

Число факторов: 5, регуляризация: 0.1
Средний RMSE по фолдам: 0.9154


### 2. Сравните результаты рекомендаций посредством коллаборативной фильтрации и факторизации матрицы рейтингов

Сравниваем RMSE на test. Значения для модели по сходству пользователей взяты из задания 2.

In [5]:
def compare(als_rmse, neighbors_rmse):
    print(f"По сходству пользователей: {neighbors_rmse:.4f}")
    print(f"ALS: {als_rmse:.4f}")
    print(f"Разница (ALS минус соседи): {als_rmse - neighbors_rmse:+.4f}")


compare(small_rmse, 0.9000)

По сходству пользователей: 0.9000
ALS: 0.9004
Разница (ALS минус соседи): +0.0004


## Полный набор MovieLens

### 1. Выберите модель ALS по минимальному значению rmse. Для этого используйте кросс-валидацию k-folds с k=4

In [ ]:
full_rmse = run_als("../ml-latest")

Число факторов: 15, регуляризация: 0.1
Средний RMSE по фолдам: 0.8178


### 2. Сравните результаты рекомендаций посредством коллаборативной фильтрации и факторизации матрицы рейтингов

In [ ]:
compare(full_rmse, 0.8772)

По сходству пользователей: 0.8772
ALS: 0.8179
Разница (ALS минус соседи): -0.0593
